# DocFusion EDA

This notebook explores the challenge's synthetic receipt dataset and summarizes the signals available for downstream extraction and forgery detection.

In [ ]:
from __future__ import annotations

import json
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

In [ ]:
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
DATA_DIR = ROOT / "dummy_data"
NOTEBOOK_DIR = ROOT / "notebooks"

def load_jsonl(path: Path) -> list[dict]:
    with path.open() as handle:
        return [json.loads(line) for line in handle]

train_records = load_jsonl(DATA_DIR / "train" / "train.jsonl")
test_records = load_jsonl(DATA_DIR / "test" / "test.jsonl")
test_labels = load_jsonl(DATA_DIR / "test" / "labels.jsonl")

len(train_records), len(test_records), len(test_labels)

In [ ]:
eda_frame = pd.DataFrame(
    {
        "id": [record["id"] for record in train_records],
        "vendor": [record["fields"]["vendor"] for record in train_records],
        "date": [record["fields"]["date"] for record in train_records],
        "total": [float(record["fields"]["total"]) for record in train_records],
        "is_forged": [record["label"]["is_forged"] for record in train_records],
        "fraud_type": [record["label"]["fraud_type"] for record in train_records],
    }
)

eda_frame.head()

In [ ]:
summary = {
    "vendor_distribution": Counter(eda_frame["vendor"]),
    "fraud_distribution": Counter(eda_frame["fraud_type"]),
    "total_stats": eda_frame["total"].describe().round(2).to_dict(),
    "forged_share": float(eda_frame["is_forged"].mean()),
}
summary

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("DocFusion Dummy Data EDA", fontsize=16)

vendor_counts = eda_frame["vendor"].value_counts().sort_index()
axes[0, 0].bar(vendor_counts.index, vendor_counts.values, color="steelblue")
axes[0, 0].set_title("Vendor distribution")
axes[0, 0].tick_params(axis="x", rotation=30)

axes[0, 1].hist(eda_frame["total"], bins=10, color="teal", edgecolor="black", alpha=0.7)
axes[0, 1].set_title("Total amount distribution")
axes[0, 1].set_xlabel("Amount")

pivot = (
    eda_frame.groupby(["vendor", "is_forged"])
    .size()
    .unstack(fill_value=0)
    .rename(columns={0: "genuine", 1: "forged"})
)
pivot.plot(kind="bar", ax=axes[1, 0], color=["#2f9e44", "#e03131"])
axes[1, 0].set_title("Forgery counts by vendor")
axes[1, 0].tick_params(axis="x", rotation=30)

fraud_counts = eda_frame.loc[eda_frame["fraud_type"] != "none", "fraud_type"].value_counts()
axes[1, 1].pie(fraud_counts.values, labels=fraud_counts.index, autopct="%1.0f%%", startangle=90)
axes[1, 1].set_title("Fraud type mix")

fig.tight_layout()
output_path = NOTEBOOK_DIR / "eda_dummy_data.png"
fig.savefig(output_path, dpi=150, bbox_inches="tight")
output_path

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
eda_frame.boxplot(column="total", by="is_forged", ax=ax)
ax.set_title("Total amount by forgery label")
ax.set_xlabel("is_forged")
ax.set_ylabel("Amount")
fig.suptitle("")
comparison_path = NOTEBOOK_DIR / "eda_amount_comparison.png"
fig.savefig(comparison_path, dpi=150, bbox_inches="tight")
comparison_path

## Notes

- The dummy training set is balanced between genuine and forged receipts.
- Fraud labels are split across `price_change`, `text_edit`, and `layout_edit`.
- Amount alone is a weak separator, which supports the project's combined OCR and image-feature approach.